In [20]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler

In [21]:
df = pd.read_csv('vehicles-data.csv')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11388 entries, 0 to 11387
Data columns (total 14 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   ID                11388 non-null  int64  
 1   VIN               7773 non-null   object 
 2   seller_rating     11388 non-null  int64  
 3   year              11286 non-null  float64
 4   manufacturer      10913 non-null  object 
 5   model             11262 non-null  object 
 6   condition         6360 non-null   object 
 7   cylinders         7654 non-null   object 
 8   fuel              11281 non-null  object 
 9   odometer          11360 non-null  float64
 10  seats             11388 non-null  int64  
 11  passenger_volume  11388 non-null  int64  
 12  posting_date      11388 non-null  object 
 13  price             11388 non-null  int64  
dtypes: float64(2), int64(5), object(7)
memory usage: 1.2+ MB


#### General Instructions:
##### our task is to predict a vehicle price using provided all collected information

##### (1) inspect the data (the csv file), make sure you understand what are your predictors and the outcome variable
##### (2) fill in missing values - ints or floats for those that can be treated as numerals; use "unknown" for missing categorical vars
##### (3) decide on columns that are obviously not useful for regression task (e.g., strings) and get rid of them

#### Question 1 -
##### (1) prepare the dataset for regression analysis (use all available predictors)
##### (1.1) for categorical variables: you may find this function useful - https://pandas.pydata.org/docs/reference/api/pandas.get_dummies.html
##### (2) train regression model with **raw (unscaled)** predictors - write down your R^2
##### -----
##### (3) inspect the data and identify multi-collinearity issues; review pedictor coeffs in (2) and understand the issues
##### (4) handle multi-collinearity issues if exist, and re-train the regression model - write down your R^2
##### (5) inspect numerical predictors and identify those that do not contribute to the model (they are useless)
##### (6) if you identified such predictors, remove and re-train the regression model - write down your R^2

#### Question 1 - solution (three R^2 reported as dataframe + short explanation as free text)

In [22]:
# 1) fill empty values per column type
# numerical -> mean
numeric_cols = df.select_dtypes(include=['number']).columns
df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].mean())

# categorical -> "unknown"
categorical_cols = df.select_dtypes(include=['object', 'category']).columns
df[categorical_cols] = df[categorical_cols].fillna('unknown')

# 1.1) cleaning columns that are not relevant and expensive for 1-HOT coding
cols_to_ignore = ['VIN', 'ID', 'posting_date']  
clean_df = df.drop(columns=cols_to_ignore)

# we take the 300 most significant modeks from the models list in order to shrink the final pd to make the fit faster.
# we choose to ignore the other models because they are relativly rare and make the fit too slow.
top_300_models = clean_df['model'].value_counts().nlargest(300).index
clean_df['model'] = clean_df['model'].where(clean_df['model'].isin(top_300_models), 'other')


############ helper functions ############
# 2) train regression model with raw (unscaled) predictors and print R^2
def model_fit_and_print_rsquared(clean_df_encoded):    
    # define predictors
    y = clean_df_encoded['price']
    x = clean_df_encoded.drop(columns=['price'])
    x = sm.add_constant(x)
    
    # fit the model
    model = sm.OLS(y, x)
    results = model.fit()
    
    # print R^2
    print("R ^ 2 =", results.rsquared)
    return results
    
def prep_1_HOT_fit_model_print_rsquared(clean_df, drop_first_status: bool):
    clean_df_encoded = pd.get_dummies(clean_df, drop_first=drop_first_status)
    clean_df_encoded = clean_df_encoded.astype(float)
    return model_fit_and_print_rsquared(clean_df_encoded)

############################################

# answer to Q2
results = prep_1_HOT_fit_model_print_rsquared(clean_df, False)

# 3) inspecting the model
print(results.summary())

# 4) fix: remove first column of 1-HOT 
results = prep_1_HOT_fit_model_print_rsquared(clean_df, True)
print(results.summary())

# 5) identify useless columns (Pvalue > 0.05), print them and remove them from the data frame.
pvals = results.pvalues
useless_columns = [col for col in numeric_cols if col in pvals.index and pvals[col] > 0.05]
print("useless columns:", useless_columns)

# 6) retrain the model without the useless data.
clean_df_step6 = clean_df.drop(columns=useless_columns)
results_final = prep_1_HOT_fit_model_print_rsquared(clean_df_step6, True)
print(results_final.summary())

R ^ 2 = 0.4942555424093883
                            OLS Regression Results                            
Dep. Variable:                  price   R-squared:                       0.494
Model:                            OLS   Adj. R-squared:                  0.478
Method:                 Least Squares   F-statistic:                     29.76
Date:                Sat, 09 May 2026   Prob (F-statistic):               0.00
Time:                        14:06:23   Log-Likelihood:            -1.2149e+05
No. Observations:               11388   AIC:                         2.437e+05
Df Residuals:                   11025   BIC:                         2.464e+05
Df Model:                         362                                         
Covariance Type:            nonrobust                                         
                                           coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------

#### Question 2 -
##### (1) re-train your final regression model from Q1 (6) with **scaled** predictors - write down your R^2

In [23]:
scaler = StandardScaler()
numeric_columns = clean_df_step6.select_dtypes(include=['number']).columns.drop('price')
scaled_df = clean_df_step6.copy()
scaled_df[numeric_columns] = scaler.fit_transform(scaled_df[numeric_columns])
result = prep_1_HOT_fit_model_print_rsquared(scaled_df, True)

R ^ 2 = 0.49421886494052114


#### Question 2 - solution (one R^2 reported as dataframe + short explanation as free text)

In [32]:
df_solution_q2 = pd.DataFrame({
    "R ^ 2": [result.rsquared], 
    "Explanation": [
        "R^2 remains the same. It doesn't improve because we didn't add features, "
        "and it doesn't decrease because Standard Scaling is a linear transformation. "
        "We standardized the predictors to make the coefficients comparable, "
        "without changing the linear relationship between them and the price."
    ]
})

display(df_solution_q2.style.hide(axis="index"))

R ^ 2,Explanation
0.494219,"R^2 remains the same. It doesn't improve because we didn't add features, and it doesn't decrease because Standard Scaling is a linear transformation. We standardized the predictors to make the coefficients comparable, without changing the linear relationship between them and the price."


#### Question 3 -
##### (1) what amount is added to a vehicle price with each additional seat?
##### (2) what amount is added to a vehicle price with each additional unit of odometer?

In [33]:
seat_coef = result.params['seats']
odometer_coef = result.params['odometer']

#### Question 3 - solution (report as a dataframe)

In [34]:
df_solution_q3 = pd.DataFrame({'seat coef': [seat_coef], 'odometer coef': [odometer_coef]})
display(df_solution_q3.style.hide(axis="index"))

seat coef,odometer coef
3102.913131,-1694.725359


#### Question 4 -
##### for your final model in Q2: write down the list of **numerical** variables that have signifnicant effect on the price:
##### for each such variable, specify if it has a positive or negative effect
##### sort your list by the magnitude of the effect (large -> small)

In [46]:
numeric_columns = clean_df_step6.select_dtypes(include=['number']).columns.drop('price')    # filter only numerical
numeric_columns_coefs = result.params[numeric_columns]                                      # list of coefs
sorted_coefs = numeric_columns_coefs.sort_values(key=abs, ascending=False)                  # sort the absolute value to determine the effect
sorted_variable_names = sorted_coefs.index                                                  # sort the variable according to the sorted coefs
coef_type = ['positive effect' if coef > 0 else 'negative effect' for coef in sorted_coefs] # determine the type of effect (+ / -)

#### Question 4 - solution (report as a dataframe)

In [47]:
df_solution_q4 = pd.DataFrame({'variable': sorted_variable_names, 'type of effect': coef_type})
display(df_solution_q4.style.hide(axis="index"))

variable,type of effect
year,positive effect
seats,positive effect
odometer,negative effect
